# News Topic Classifier Using BERT

## Project Overview

This project is a News Topic Classification system developed using the BERT transformer model. The model classifies news headlines into four categories: World, Sports, Business, and Sci/Tech. The AG News dataset was used for training and evaluation.

## Objective

The main objective of this project is to build an accurate Natural Language Processing (NLP) model capable of automatically identifying the category of a news headline using deep learning and transformer-based techniques.

## Dataset

* Dataset Name: AG News Dataset
* Categories:

  * World
  * Sports
  * Business
  * Sci/Tech

## Technologies Used

* Python
* Hugging Face Transformers
* BERT (bert-base-uncased)
* Scikit-learn
* Gradio
* Google Colab

## Methodology

1. Loaded and preprocessed the AG News dataset.
2. Tokenized text using the BERT tokenizer.
3. Fine-tuned the `bert-base-uncased` model for sequence classification.
4. Evaluated the model using Accuracy and F1-score.
5. Generated a confusion matrix for performance analysis.
6. Deployed the trained model using Gradio for real-time predictions.

## Results

The model achieved strong classification performance with high accuracy and F1-score. The confusion matrix showed that most predictions were correctly classified across all categories.

## Deployment

A Gradio-based web interface was created where users can enter a news headline and receive the predicted category along with confidence score.

## Conclusion

This project demonstrates the practical implementation of transformer-based NLP models for text classification tasks and highlights the effectiveness of BERT in news topic prediction.


**Install Libraries**

In [ ]:
!pip install transformers datasets accelerate evaluate scikit-learn -q

**Import Libraries**

In [ ]:
from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    pipeline
)

import numpy as np
from sklearn.metrics import accuracy_score, f1_score

**Load Dataset**

In [ ]:
dataset = load_dataset("ag_news")

print(dataset)

**Check Sample Data**

In [ ]:
print(dataset["train"][0])

**Show Category Names**

In [ ]:
label_names = dataset["train"].features["label"].names

print(label_names)

**Load BERT Tokenizer**

In [ ]:
model_checkpoint = "bert-base-uncased"

tokenizer = AutoTokenizer.from_pretrained(model_checkpoint)


Tokenization

In [ ]:
def tokenize_function(example):
    return tokenizer(
        example["text"],
        padding="max_length",
        truncation=True,
        max_length=128
    )

tokenized_dataset = dataset.map(tokenize_function, batched=True)

**Load BERT Model**

In [ ]:
model = AutoModelForSequenceClassification.from_pretrained(
    model_checkpoint,
    num_labels=4
)

**Metrics Function**

In [ ]:
def compute_metrics(eval_pred):

    logits, labels = eval_pred

    predictions = np.argmax(logits, axis=-1)

    acc = accuracy_score(labels, predictions)

    f1 = f1_score(labels, predictions, average="weighted")

    return {
        "accuracy": acc,
        "f1": f1
    }

**Training Arguments**

In [ ]:
training_args = TrainingArguments(
    output_dir="./results",

    # evaluation_strategy="epoch", # Temporarily removed to resolve TypeError

    save_strategy="epoch",

    learning_rate=2e-5,

    per_device_train_batch_size=16,

    per_device_eval_batch_size=16,

    num_train_epochs=2,

    weight_decay=0.01,

    logging_dir="./logs",

    load_best_model_at_end=False # Changed to False to prevent ValueError when no evaluation strategy is defined
)

**Trainer Setup**

In [ ]:
trainer = Trainer(
    model=model,

    args=training_args,

    train_dataset=tokenized_dataset["train"]
        .shuffle(seed=42)
        .select(range(10000)),

    eval_dataset=tokenized_dataset["test"]
        .shuffle(seed=42)
        .select(range(2000)),

    compute_metrics=compute_metrics
)

**Start Training**

In [ ]:
trainer.train()

In [ ]:
results = trainer.evaluate()

print(results)

**Add Classification Report**

In [ ]:
from sklearn.metrics import classification_report

predictions = trainer.predict(
    tokenized_dataset["test"].select(range(2000))
)

y_pred = np.argmax(predictions.predictions, axis=1)

y_true = predictions.label_ids

print(classification_report(y_true, y_pred))

**Add Confusion Matrix Visualization**

In [ ]:
from sklearn.metrics import confusion_matrix
import matplotlib.pyplot as plt
import numpy as np

cm = confusion_matrix(y_true, y_pred)

labels = ["World", "Sports", "Business", "Sci/Tech"]

plt.figure(figsize=(7,6))

plt.imshow(cm, cmap="Blues")

plt.title("Confusion Matrix")

plt.xlabel("Predicted Label")
plt.ylabel("True Label")

plt.xticks(np.arange(len(labels)), labels)
plt.yticks(np.arange(len(labels)), labels)

plt.colorbar()

for i in range(len(labels)):
    for j in range(len(labels)):
        plt.text(j, i, cm[i, j],
                 ha="center",
                 color="black")

plt.show()

In [ ]:
from transformers import pipeline

# Ensure the model and tokenizer are saved before loading them
model.save_pretrained("saved_model")
tokenizer.save_pretrained("saved_model")

classifier = pipeline(
    "text-classification",
    model="saved_model",
    local_files_only=True
)

result = classifier(
    "Apple launches new AI-powered iPhone"
)

print(result)

**Install Gradio**

In [ ]:
!pip install gradio -q

**Create App**

In [ ]:
import gradio as gr
from transformers import pipeline

# Load saved model
classifier = pipeline(
    task="text-classification",
    model="./saved_model",
    tokenizer="./saved_model"
)

# Label mapping
labels = {
    "LABEL_0": "World",
    "LABEL_1": "Sports",
    "LABEL_2": "Business",
    "LABEL_3": "Sci/Tech"
}

# Prediction function
def predict_news(text):

    if text.strip() == "":
        return "Please enter a news headline."

    result = classifier(text)[0]

    label = labels.get(result["label"], "Unknown")

    score = round(result["score"] * 100, 2)

    return f"""
📰 Category: {label}

📊 Confidence: {score}%
"""

# Gradio Interface
app = gr.Interface(
    fn=predict_news,

    inputs=gr.Textbox(
        lines=3,
        placeholder="Enter news headline here..."
    ),

    outputs=gr.Textbox(),

    title="📰 News Topic Classifier",

    description="BERT-based AG News Classification System",

    theme="soft"
)

# Launch App
app.launch(share=True)

In [ ]:
model.save_pretrained("saved_model")
tokenizer.save_pretrained("saved_model")

In [ ]:
import shutil

shutil.make_archive("saved_model", 'zip', "saved_model")

In [ ]:
from google.colab import files
files.download("saved_model.zip")